# Nome: Davi dos Santos Mattos
# DRE: 119133049

# Função retirada das notas de aula: "Otimização Escalar e Vetorial - Volume 2: Otimização Escalar" - Professor: Ricardo H. C. Takahashi

> f(x) = 2x1\^2 + x2^2 + 2x1x2 + x1 - 2x2 + 3 (Pág. 111 Ex: 6.8)



In [24]:
import numpy as np

def f(X):
    return 2*X[0]**2 + X[1]**2 + 2*X[0]*X[1] + X[0] - 2*X[1] + 3

Utilizando a biblioteca `jax.numpy` para calcular numéricamente



In [25]:
import jax.numpy as jnp
from jax import grad

def f(X):
    return 2*X[0]**2 + X[1]**2 + 2*X[0]*X[1] + X[0] - 2*X[1] + 3

def grad_f(x_zero):
  grad_f = grad(f)
  return grad_f(jnp.array(x_zero))

result = np.array(grad_f([-1.,1.]))
print(result)

[-1. -2.]


Utilizando a biblioteca `sympy` para calcular simbólicamente

In [26]:
import sympy as sp
x1, x2 = sp.symbols('x1 x2')
f = 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3

grad = [sp.diff(f, var) for var in (x1, x2)]
print(grad)

[4*x1 + 2*x2 + 1, 2*x1 + 2*x2 - 2]


In [27]:
x0 = {x1: -0.75, x2: 1.5}
g0 = [g.subs(x0) for g in grad]
print("Gradiente em x0:", g0)

Gradiente em x0: [1.00000000000000, -0.500000000000000]


In [28]:
alpha = sp.symbols('alpha')

x1_alpha = x0[x1] - alpha * g0[0]
x2_alpha = x0[x2] - alpha * g0[1]

print("x1(alpha):", x1_alpha)
print("x2(alpha):", x2_alpha)

x1(alpha): -1.0*alpha - 0.75
x2(alpha): 0.5*alpha + 1.5


In [29]:
f_alpha = f.subs({
    x1: x1_alpha,
    x2: x2_alpha
})

print("f(alpha):")
sp.expand(f_alpha)

f(alpha):


1.25*alpha**2 - 1.25*alpha + 0.375

In [30]:
df_alpha = sp.diff(f_alpha, alpha)
alpha_star = sp.solve(df_alpha, alpha)
print(df_alpha)
print(alpha_star)

2.5*alpha - 1.25
[0.500000000000000]


In [31]:
x1_new = x0[x2] - alpha_star[0] * g0[1]
x1_new

1.75000000000000

## Criando algoritmo do gradiente "analítico" (com alpha dinâmico)

### Algoritmo Simbólico

In [ ]:
import numpy as np
import sympy as sp
import math

x1, x2 = sp.symbols('x1 x2')
alpha = sp.symbols('alpha', positive=True, real=True)

def f_symb():
    x1, x2 = sp.symbols('x1 x2')
    f = 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3
    return f


def grad_analitico_symb(func, x_zero, epsilon=1e-3, max_iterations=5000):
  x0 = {x1: x_zero[0], x2: x_zero[1]}
  trajetoria = [x_zero.copy()]
  grad = [sp.diff(func, var) for var in (x1, x2)]


  for i in range(max_iterations):

    grad_x0 = [g.subs(x0) for g in grad]

    # Critério de parada: norma do gradiente
    if np.linalg.norm([float(g) for g in grad_x0]) < epsilon:
      return x0, i, np.array(trajetoria)

    x_alpha = sp.Matrix([x0[x1], x0[x2]]) - alpha * sp.Matrix(grad_x0)

    f_alpha = func.subs({
      x1: x_alpha[0],
      x2: x_alpha[1]
    })


    df_alpha = sp.diff(f_alpha, alpha)
    alpha_star = sp.solve(df_alpha, alpha)

    alpha_val = None

    # Tenta primeiro a solução na posição 0 para manter o comportamento original.
    if len(alpha_star) > 0 and sp.im(alpha_star[0]) == 0:
      alpha_val = sp.re(alpha_star[0])
    else:
      # Se alpha_star[0] for complexo, tenta outra raiz real.
      for a in alpha_star[1:]:
        if sp.im(a) == 0:
          alpha_val = sp.re(a)
          break

    # Se não houver raiz real, encerra a iteração com mensagem.
    if alpha_val is None:
      print(f"Iteração {i}: nenhuma raiz real para alpha. Encerrando.")
      return x0, i, np.array(trajetoria)

    x0_new = sp.Matrix([x0[x1], x0[x2]]) - alpha_val * sp.Matrix(grad_x0)
    x0 = {x1: x0_new[0], x2: x0_new[1]}
    trajetoria.append([round(float(x0[x1]), 5), round(float(x0[x2]),5)])


    #print(f"ite: {i} | x = {x0} | f(x) = {round(float(func.subs({x1: x0[x1], x2: x0[x2]})), 4)}")

In [33]:
# Exemplo retirado das notas de aula Pág. 111 Exemplo 6.8

x_min, ite , f_traj_symb = grad_analitico_symb(f_symb(), [-1,1])
print(f'x* = {x_min}')
print(f'Iterações: {ite}')

ite: 0 | x = {x1: -3/4, x2: 3/2} | f(x) = 0.375
ite: 1 | x = {x1: -5/4, x2: 7/4} | f(x) = 0.0625
ite: 2 | x = {x1: -9/8, x2: 2} | f(x) = -0.0938
ite: 3 | x = {x1: -11/8, x2: 17/8} | f(x) = -0.1719
ite: 4 | x = {x1: -21/16, x2: 9/4} | f(x) = -0.2109
ite: 5 | x = {x1: -23/16, x2: 37/16} | f(x) = -0.2305
ite: 6 | x = {x1: -45/32, x2: 19/8} | f(x) = -0.2402
ite: 7 | x = {x1: -47/32, x2: 77/32} | f(x) = -0.2451
ite: 8 | x = {x1: -93/64, x2: 39/16} | f(x) = -0.2476
ite: 9 | x = {x1: -95/64, x2: 157/64} | f(x) = -0.2488
ite: 10 | x = {x1: -189/128, x2: 79/32} | f(x) = -0.2494
ite: 11 | x = {x1: -191/128, x2: 317/128} | f(x) = -0.2497
ite: 12 | x = {x1: -381/256, x2: 159/64} | f(x) = -0.2498
ite: 13 | x = {x1: -383/256, x2: 637/256} | f(x) = -0.2499
ite: 14 | x = {x1: -765/512, x2: 319/128} | f(x) = -0.25
ite: 15 | x = {x1: -767/512, x2: 1277/512} | f(x) = -0.25
ite: 16 | x = {x1: -1533/1024, x2: 639/256} | f(x) = -0.25
ite: 17 | x = {x1: -1535/1024, x2: 2557/1024} | f(x) = -0.25
ite: 18 | x =

### Algoritmo Numérico

In [34]:
import jax
import jax.numpy as jnp
import numpy as np

def f(x):
    """Função objetivo: 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3"""
    x1, x2 = x[0], x[1]
    return 2*x1**2 + x2**2 + 2*x1*x2 + x1 - 2*x2 + 3

def grad_analitico_num(func, x_zero, epsilon=1e-3, max_iterations=5000):
    # Definir gradiente automático do JAX
    grad_func = jax.grad(func)

    x0 = jnp.array(x_zero, dtype=float)
    trajetoria = [x_zero.copy()]

    for i in range(max_iterations):
        # Calcular gradiente no ponto atual
        grad_x0 = grad_func(x0)

        # Critério de parada: norma do gradiente
        if jnp.linalg.norm(grad_x0) < epsilon:
            return {"x1": float(x0[0]), "x2": float(x0[1])}, i, trajetoria

        # Busca linear por otimização: encontrar melhor alpha
        def f_alpha(alpha):
            return func(x0 - alpha * grad_x0)

        # Usar otimização numérica para encontrar alpha optimal
        from scipy.optimize import minimize_scalar

        result = minimize_scalar(f_alpha, bounds=(0, 10), method='bounded')
        alpha_star = result.x

        # Garantir que alpha é válido
        if alpha_star < 1e-10:
            alpha_star = 0.01

        # Atualizar x0
        x0_new = x0 - alpha_star * grad_x0
        x0 = x0_new
        trajetoria.append([round(float(x0[0]), 5), round(float(x0[1]), 5)])

        print(f"ite: {i+1} | x = {{'x1': {float(x0[0]):.6f}, 'x2': {float(x0[1]):.6f}}} | f(x) = {round(float(func(x0)), 4)}")

    return {"x1": float(x0[0]), "x2": float(x0[1])}, max_iterations, trajetoria

Obs: O código acima foi gerado a partir da "versão simbólica" com uso de LLM

In [35]:
x_min, ite , f_traj_num = grad_analitico_num(f, np.array([-1,1]))
print(f'x* = {x_min}')
print(f'Iterações: {ite}')

ite: 1 | x = {'x1': -0.750039, 'x2': 1.499922} | f(x) = 0.375
ite: 2 | x = {'x1': -1.250183, 'x2': 1.750189} | f(x) = 0.0624
ite: 3 | x = {'x1': -1.125166, 'x2': 2.000043} | f(x) = -0.0938
ite: 4 | x = {'x1': -1.375781, 'x2': 2.125619} | f(x) = -0.172
ite: 5 | x = {'x1': -1.312981, 'x2': 2.250359} | f(x) = -0.2111
ite: 6 | x = {'x1': -1.438159, 'x2': 2.313375} | f(x) = -0.2306
ite: 7 | x = {'x1': -1.406798, 'x2': 2.375547} | f(x) = -0.2403
ite: 8 | x = {'x1': -1.469737, 'x2': 2.407296} | f(x) = -0.2452
ite: 9 | x = {'x1': -1.453785, 'x2': 2.438251} | f(x) = -0.2476
ite: 10 | x = {'x1': -1.484870, 'x2': 2.453990} | f(x) = -0.2488
ite: 11 | x = {'x1': -1.476969, 'x2': 2.469481} | f(x) = -0.2494
ite: 12 | x = {'x1': -1.492057, 'x2': 2.476750} | f(x) = -0.2497
ite: 13 | x = {'x1': -1.488324, 'x2': 2.484509} | f(x) = -0.2498
ite: 14 | x = {'x1': -1.496316, 'x2': 2.488388} | f(x) = -0.2499
ite: 15 | x = {'x1': -1.494208, 'x2': 2.492327} | f(x) = -0.25
ite: 16 | x = {'x1': -1.497966, 'x2': 2.

O uso da seguinte seção


```
# Usar otimização numérica para encontrar alpha optimal
        from scipy.optimize import minimize_scalar

        result = minimize_scalar(f_alpha, bounds=(0, 10), method='bounded')
        alpha_star = result.x

        # Garantir que alpha é válido
        if alpha_star < 1e-10:
            alpha_star = 0.01
```
Acaba reduzindo ligeiramente o custo computacional e acrescentando um pouco mais de exatidão, que pode ser visto na diferença de iterações e no x*


## Comparação entre algoritmos

### Para F(x)

In [36]:
# Implementação do Método do Gradiente feita pela profª

def metodo_gradiente(J_grad, x_zero, alpha=0.002, epsilon=1e-2, max_iterations=5000):
    x = np.array(x_zero)
    trajetoria = [x.copy()]
    for i in range(max_iterations):
        grad = J_grad(x)
        x = x - alpha * grad # Passo de atualização: x_novo = x - alpha * grad
        trajetoria.append(x)

        # Critério de parada: norma do gradiente
        if np.linalg.norm(grad) < epsilon:
            return x, i + 1, np.array(trajetoria)

    return x, max_iterations, np.array(trajetoria)

In [37]:
def f_grad(X):
  x1, x2 = X[0], X[1]
  dx = 4*x1 + 2*x2 + 1
  dy = 2*x1 + 2*x2 - 2
  return np.array([dx, dy])


x_zero = [-1.,1.]
x_min, it, f_trajetoria_num = metodo_gradiente(f_grad, x_zero, alpha=0.002, max_iterations=5000)

print(f'x_zero = {x_zero}')
print(f'x* = {x_min}')
print(f'F(x*) = {f(x_min)}')
print(f'Gradiente F(x*) = {f_grad(x_min)}')
print(f'Iterações: {it}')

x_zero = [-1.0, 1.0]
x* = [-1.49313272  2.48888851]
F(x*) = -0.24993482716472304
Gradiente F(x*) = [ 0.00524613 -0.00848842]
Iterações: 3119


In [38]:
x_zero = [-1,1]
x_min, ite , f_traj_symb = grad_analitico_symb(f_symb(), x_zero)
fx = f_symb().subs({
      x1: x_min[x1],
      x2: x_min[x2]
    })
print(f'x* = {x_min}')
print(f"""F(x*) = {float(fx)}""")
print(f'Iterações: {ite}')

ite: 0 | x = {x1: -3/4, x2: 3/2} | f(x) = 0.375
ite: 1 | x = {x1: -5/4, x2: 7/4} | f(x) = 0.0625
ite: 2 | x = {x1: -9/8, x2: 2} | f(x) = -0.0938
ite: 3 | x = {x1: -11/8, x2: 17/8} | f(x) = -0.1719
ite: 4 | x = {x1: -21/16, x2: 9/4} | f(x) = -0.2109
ite: 5 | x = {x1: -23/16, x2: 37/16} | f(x) = -0.2305
ite: 6 | x = {x1: -45/32, x2: 19/8} | f(x) = -0.2402
ite: 7 | x = {x1: -47/32, x2: 77/32} | f(x) = -0.2451
ite: 8 | x = {x1: -93/64, x2: 39/16} | f(x) = -0.2476
ite: 9 | x = {x1: -95/64, x2: 157/64} | f(x) = -0.2488
ite: 10 | x = {x1: -189/128, x2: 79/32} | f(x) = -0.2494
ite: 11 | x = {x1: -191/128, x2: 317/128} | f(x) = -0.2497
ite: 12 | x = {x1: -381/256, x2: 159/64} | f(x) = -0.2498
ite: 13 | x = {x1: -383/256, x2: 637/256} | f(x) = -0.2499
ite: 14 | x = {x1: -765/512, x2: 319/128} | f(x) = -0.25
ite: 15 | x = {x1: -767/512, x2: 1277/512} | f(x) = -0.25
ite: 16 | x = {x1: -1533/1024, x2: 639/256} | f(x) = -0.25
ite: 17 | x = {x1: -1535/1024, x2: 2557/1024} | f(x) = -0.25
ite: 18 | x =

### Para Função Zakharov

In [39]:
x1, x2 = sp.symbols('x1 x2')

def zakharov_symb(): # [-5, 10]
  xi = [x1, x2]
  return sum(xi[i]**2 for i in range(len(xi))) + \
         sum(0.5 * xi[j] for j in range(len(xi)))**2 + \
         sum(0.5 * xi[k] for k in range(len(xi)))**4

x_zero = [-10,10]
x_min, ite , zakharov_traj_symb = grad_analitico_symb(zakharov_symb(), x_zero)

fx = zakharov_symb().subs({
      x1: x_min[x1],
      x2: x_min[x2]
})


print(f'x* = {x_min}')
print(f"""Zakharov(x*) = {float(fx)}""")
print(f'Iterações: {ite}')

ite: 0 | x = {x1: 0, x2: 0} | f(x) = 0.0
x* = {x1: 0, x2: 0}
Zakharov(x*) = 0.0
Iterações: 1


In [40]:
def zakharov_num(X):
  return sum(X**2) + sum(0.5 * X) ** 2 + sum(0.5 * X) ** 4
def zakharov_grad(X):
  return 0.25*X**3 + 2.5*X

x_zero = [-10.,10.]
x_min, it, zakharov_trajetoria_num = metodo_gradiente(zakharov_grad, x_zero, alpha=0.002, max_iterations=5000)

print(f'x_zero = {x_zero}')
print(f'x* = {x_min}')
print(f'F(x*) = {zakharov_num(x_min)}')
print(f'Gradiente F(x*) = {zakharov_grad(x_min)}')
print(f'Iterações: {it}')

x_zero = [-10.0, 10.0]
x* = [-0.00280043  0.00280043]
F(x*) = 1.5684855975870023e-05
Gradiente F(x*) = [-0.00700109  0.00700109]
Iterações: 1391


### Para função Rosenbrock

In [41]:
x1, x2 = sp.symbols('x1 x2')

def rosenbrock_symb():
  return 100 * (x2 - x1**2)**2 + (x1 - 1)**2

def rosenbrock(X):
  return np.sum(100 * (X[1:] - X[:-1]**2)**2 + (X[:-1] - 1)**2, axis=0)

def rosenbrock_gradiente(X):
  grad = np.zeros_like(X)
  termo_interno = X[1:] - X[:-1]**2

In [42]:
x_zero = [-10,10]
x_min, ite , rosenbrock_traj_symb = grad_analitico_symb(rosenbrock_symb(), x_zero)

fx = rosenbrock_symb().subs({
      x1: x_min[x1],
      x2: x_min[x2]
})


print(f'x* = {x_min}')
print(f"""Rosenbrock(x*) = {float(fx)}""")
print(f'Iterações: {ite}')

ite: 0 | x = {x1: 4500/180011 + sqrt(408118585691274)*cos(atan(sqrt(1888237236889810758956696514593328162342534)/511852630031714790)/3)/5400330, x2: 300*sqrt(408118585691274)*cos(atan(sqrt(1888237236889810758956696514593328162342534)/511852630031714790)/3)/32403960121 + 340281091210/32403960121} | f(x) = 5.1311
Iteração 1: nenhuma raiz real para alpha. Encerrando.
x* = {x1: 4500/180011 + sqrt(408118585691274)*cos(atan(sqrt(1888237236889810758956696514593328162342534)/511852630031714790)/3)/5400330, x2: 300*sqrt(408118585691274)*cos(atan(sqrt(1888237236889810758956696514593328162342534)/511852630031714790)/3)/32403960121 + 340281091210/32403960121}
Rosenbrock(x*) = 5.131089606610304
Iterações: 1


## Plotando o Gráfico


### Para F(x)

In [43]:
x= np.linspace(min(f_trajetoria_num[:,0])-1, max(f_trajetoria_num[:,1])-2, 100)
y= np.linspace(min(f_trajetoria_num[:,1])-0.5, max(f_trajetoria_num[:,1])+0.5, 100)

X, Y = np.meshgrid(x, y)
Z = f(np.array([X, Y]))

fig = plt.figure(figsize=(20, 10))
ax = plt.axes(projection='3d')
ax.set_title('F(x)')
ax.view_init(elev=50., azim=30)
s = ax.plot_surface(X, Y, Z, rstride=1, cstride=1, cmap='jet', edgecolor='none')
fig.colorbar(s, shrink=0.5, aspect=5)

# Usando função contour
fig, ax = plt.subplots()
contourFx = ax.contour(X,Y,Z)
ax.set_title("Testando função contour para F(x)")

NameError: name 'plt' is not defined

### Para Função []

## Plotando o Gradiente

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

### Para F(x)

In [ ]:
x= np.linspace(min(f_trajetoria_num[:,0])-1, max(f_trajetoria_num[:,1])-2, 100)
y= np.linspace(min(f_trajetoria_num[:,1])-0.5, max(f_trajetoria_num[:,1])+0.5, 100)

X, Y = np.meshgrid(x, y)
Z = f(np.array([X, Y]))

fig = plt.figure(figsize=(20, 10))
ax = plt.axes(projection='3d')
ax.set_title('F(x)')
ax.view_init(elev=50., azim=30)
s = ax.plot_surface(X, Y, Z, rstride=1, cstride=1, cmap='jet', edgecolor='none', alpha=0.7)
z_traj_num = f(np.array([f_trajetoria_num[:,0], f_trajetoria_num[:,1]]))
ax.plot3D(f_trajetoria_num[:,0], f_trajetoria_num[:,1], z_traj_num, 'ro-', label='Numérica', linewidth=1) # Uso de LLM
z_traj_symb = f(np.array([f_traj_symb[:,0], f_traj_symb[:,1]]))
ax.plot3D(f_traj_symb[:,0], f_traj_symb[:,1], z_traj_symb, 'bo-', label='Simbólica',linewidth=3) # Uso de LLM
fig.colorbar(s, shrink=0.5, aspect=5)

# Usando função contour
fig, ax = plt.subplots()
contourFx = ax.contour(X,Y,Z)
ax.set_title("Testando função contour para F(x)")
plt.plot(f_trajetoria_num[:,0], f_trajetoria_num[:,1], 'ro-', label='Numérica')
plt.plot(f_traj_symb[:,0], f_traj_symb[:,1], 'bo-', label='Simbólica')

In [ ]:
x_vals = np.linspace(min(f_trajetoria_num[:,0])-1, max(f_trajetoria_num[:,1])-2, 100)
y_vals = np.linspace(min(f_trajetoria_num[:,1])-0.5, max(f_trajetoria_num[:,1])+0.5, 100)
X, Y = np.meshgrid(x_vals, y_vals)
Z = f(np.array([X, Y]))

# Plot
plt.figure(figsize=(6,5))
plt.contour(X, Y, Z, levels=30)
plt.plot(f_trajetoria_num[:,0], f_trajetoria_num[:,1], 'ro-', label='trajetória')
plt.plot(f_traj_symb[:,0], f_traj_symb[:,1], 'bo-', label='trajetória')

plt.title(f"Descida do Gradiente - Numérico")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.show()

### Para Função